In [1]:
import os
import numpy as np
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from rasterio.features import shapes
import tools
import sys
sys.path.append('/Applications/WhiteboxTools_darwin_m_series/WBT')
from whitebox_tools import WhiteboxTools
from shapely.geometry import shape, Point
from dotenv import load_dotenv

load_dotenv()
krycklan_data_path = os.getenv('KRYCKLAN_DATA_PATH')
krycklan_gis_data_path = os.getenv('KRYCKLAN_GIS_DATA_PATH')

wbt = WhiteboxTools()

In [2]:
# PARAMETERS
resos = [20, 40, 80, 160]

# Pour points per catchment (SWEREF99TM / EPSG:3006), taken from QGIS
subcatch_points = {
    16: [(736316.5, 7128178.8), (734678.4, 7130518.4), (734695.0, 7129860.7)],
    15: [(734399.0, 7133437.0), (734017.9, 7135261.4), (734031.7, 7135296.5)],
    14: [(731321.4, 7130899.1), (729140.0, 7132681.7), (729159.0, 7132700.0)],
}
SUBCATCH_CRS = "EPSG:3006"
SNAP_DIST = 200  # metres — increase if snapping misses the channel

combined_streams_ditches_fp = os.path.join(
    krycklan_data_path, 'GIS', 'CATCHMENTS', 'ManualStreams', 'PROCESSED',
    '5haStreams_ManuallyDigitizedDitchesKrycklan_merged_5.shp'
)
local_output_root = krycklan_gis_data_path


In [3]:
for reso in resos:
    for catch, pts in subcatch_points.items():

        parent_dir = os.path.join(local_output_root, f"{reso}m", f"C{catch}")

        # flowp_d8.tif and flowacc_sca_d8.tif are the two TIFs dem_multi keeps
        flow_pointer_d8 = os.path.join(parent_dir, "flowp_d8.tif")
        flow_acc_sca_d8 = os.path.join(parent_dir, "flowacc_sca_d8.tif")

        missing = [fp for fp in [flow_pointer_d8, flow_acc_sca_d8] if not os.path.exists(fp)]
        if missing:
            print(f"  SKIP C{catch} @ {reso}m — missing: {[os.path.basename(p) for p in missing]}")
            continue

        print(f"\nProcessing C{catch} @ {reso}m ...")

        # --- Build pour-points shapefile ---
        with rasterio.open(flow_pointer_d8) as src:
            dem_crs = src.crs

        gdf_pts = gpd.GeoDataFrame(
            {"id": list(range(1, len(pts) + 1)),
             "geometry": [Point(x, y) for x, y in pts]},
            crs=SUBCATCH_CRS,
        ).to_crs(dem_crs)

        pour_pts_fp = os.path.join(parent_dir, "pour_pts.shp")
        snapped_fp  = os.path.join(parent_dir, "pour_pts_snapped.shp")
        wshed_fp    = os.path.join(parent_dir, "subcatchments.tif")

        gdf_pts.to_file(pour_pts_fp)

        # Snap to SCA (same spatial pattern as CA, works equally well for snapping)
        wbt.snap_pour_points(pour_pts_fp, flow_acc_sca_d8, snapped_fp, snap_dist=SNAP_DIST)
        wbt.watershed(flow_pointer_d8, snapped_fp, wshed_fp)

        with rasterio.open(wshed_fp) as wsrc:
            wshed           = wsrc.read(1)
            wshed_nodata    = wsrc.nodata if wsrc.nodata is not None else 0
            wshed_transform = wsrc.transform
            wshed_crs       = wsrc.crs

        sub_ids = sorted([int(v) for v in np.unique(wshed) if v != wshed_nodata and v > 0])
        print(f"  Found {len(sub_ids)} subcatchments: {sub_ids}")

        # ASC files that exist in the parent dir after dem_multi cleanup
        terrain_ascs = [
            "dem_clip_culv_fill_streams.asc",
            "twi_d8.asc",
            "twi_dinf.asc",
            "dem_clip_no_deps.asc",
            "dem_clip.asc",
            "flowacc_sca_dinf.asc",
            "flowacc_sca_d8.asc",
            "flowp_d8.asc",
            "flowp_dinf.asc",
            "slope.asc",
        ]

        for sub_id in sub_ids:
            sub_out_dir = os.path.join(local_output_root, f"{reso}m", f"C{catch}_{sub_id}")
            os.makedirs(sub_out_dir, exist_ok=True)

            # Subcatchment polygon for masking
            sub_mask_arr = (wshed == sub_id).astype("uint8")
            sub_polys = [
                shape(geom)
                for geom, val in shapes(sub_mask_arr, mask=sub_mask_arr, transform=wshed_transform)
                if val == 1
            ]
            sub_gdf = gpd.GeoDataFrame(geometry=sub_polys, crs=wshed_crs).dissolve()
            sub_gdf.to_file(os.path.join(sub_out_dir, "subcatch_extent.shp"))

            # Clip each terrain ASC to subcatchment and save as ASC
            for asc_name in terrain_ascs:
                asc_src_path = os.path.join(parent_dir, asc_name)
                if not os.path.exists(asc_src_path):
                    print(f"    ⚠️ Missing: {asc_name}")
                    continue
                asc_dst_path = os.path.join(sub_out_dir, asc_name)
                with rasterio.open(asc_src_path) as src:
                    out_image, out_transform = mask(src, sub_gdf.geometry, crop=True, nodata=-9999)
                    data = out_image[0].astype(np.float32)
                    if src.nodata is not None:
                        data = np.where(data == src.nodata, -9999, data)
                    profile = src.profile.copy()
                    profile.update({
                        "driver": "AAIGrid", "dtype": "float32", "nodata": -9999,
                        "height": out_image.shape[1], "width": out_image.shape[2],
                        "transform": out_transform, "count": 1, "crs": None,
                    })
                with rasterio.open(asc_dst_path, "w", **profile) as dst:
                    dst.write(data, 1)

            # Rasterize channels using the clipped dem_clip.asc as reference
            tools.rasterize_shapefile(
                shapefile=combined_streams_ditches_fp,
                burn_field="VALUE",
                subset=None,
                out_fp=os.path.join(sub_out_dir, "channels.asc"),
                ref_raster=os.path.join(sub_out_dir, "dem_clip.asc"),
                plot=False,
                save_in="asc",
            )

            # cmask (parent catch ID) and cmask_bi
            with rasterio.open(os.path.join(sub_out_dir, "dem_clip.asc")) as src:
                dem_data   = src.read(1)
                src_nodata = src.nodata if src.nodata is not None else -9999
                meta = src.profile.copy()
            cmask_arr    = np.where(dem_data != src_nodata, catch, -9999).astype(np.float32)
            cmask_bi_arr = np.where(dem_data != src_nodata, 1,     -9999).astype(np.float32)
            meta.update({"driver": "AAIGrid", "dtype": "float32", "nodata": -9999, "crs": None})
            with rasterio.open(os.path.join(sub_out_dir, "cmask.asc"),    "w", **meta) as dst:
                dst.write(cmask_arr, 1)
            with rasterio.open(os.path.join(sub_out_dir, "cmask_bi.asc"), "w", **meta) as dst:
                dst.write(cmask_bi_arr, 1)

            print(f"    C{catch}_{sub_id} done.")

        # Clean up temp files in parent dir
        for fname in ["pour_pts.shp", "pour_pts.dbf", "pour_pts.shx", "pour_pts.prj", "pour_pts.cpg",
                      "pour_pts_snapped.shp", "pour_pts_snapped.dbf", "pour_pts_snapped.shx",
                      "pour_pts_snapped.prj", "pour_pts_snapped.cpg",
                      "subcatchments.tif"]:
            fp = os.path.join(parent_dir, fname)
            if os.path.exists(fp):
                os.remove(fp)



Processing C16 @ 20m ...
./whitebox_tools --run="SnapPourPoints" --pour_pts='/Users/jpnousu/Data/Krycklan_GIS_data/20m/C16/pour_pts.shp' --flow_accum='/Users/jpnousu/Data/Krycklan_GIS_data/20m/C16/flowacc_sca_d8.tif' --output='/Users/jpnousu/Data/Krycklan_GIS_data/20m/C16/pour_pts_snapped.shp' --snap_dist='200' -v --compress_rasters=False

*****************************
* Welcome to SnapPourPoints *
* Powered by WhiteboxTools  *
* www.whiteboxgeo.com       *
*****************************
Reading data...
Progress: 0%
Progress: 50%
Progress: 100%
Saving data...
Output file written
Elapsed Time (excluding I/O): 0.0s
./whitebox_tools --run="Watershed" --d8_pntr='/Users/jpnousu/Data/Krycklan_GIS_data/20m/C16/flowp_d8.tif' --pour_pts='/Users/jpnousu/Data/Krycklan_GIS_data/20m/C16/pour_pts_snapped.shp' --output='/Users/jpnousu/Data/Krycklan_GIS_data/20m/C16/subcatchments.tif' -v --compress_rasters=False

****************************
* Welcome to Watershed     *
* Powered by WhiteboxTools *
* 